# Mitu's Kacchi Diary: Starter Notebook

> **TL;DR:** Load the data -> look at it -> train a model -> make a submission. The story is just for fun. If you want to skip it, run the code cells from top to bottom and you'll have a submission at the end. 

---

**Day 1.** Hi! I'm **Mitu**, I'm in class 7, and I'm learning Machine Learning.

In Bangladesh, a wedding is not one party. It's at least three: **Gaye Holud** (the turmeric party), **Biye** (the wedding day), and **Bou-bhat** (the reception). And every single one has **kacchi biryani**: rice, mutton and the most important thing in the whole world, **the aloo** (potato).

My friends and I have been going to a LOT of weddings. Some we were invited to. Some… we kind of just walked in. (In Bangladesh, weddings are so big that a few extra hungry kids with a clipboard barely get noticed.)

At every wedding we wrote down one big question:

### Did the guests go back for a second plate? 

My team:
- **Tutul**, my little cousin. He counts potatoes on plates. He is… *mostly* accurate.
- **Rafi**, who says his "MEGA AI" can solve anything. He thinks the fairy lights are the secret.
- **Nodi**, who always asks *"but WHY?"* Annoying. Also usually right.

Your job: help us predict **seconds or no seconds** for weddings we haven't seen yet. Let's go!

## Day 2: Opening the notebook (the data)

In [31]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# This finds the files whether you're on Kaggle or on your own computer
def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) or glob.glob(f"**/{name}", recursive=True)
    return hits[0]

train = pd.read_csv(find("train.csv"))
test = pd.read_csv(find("test.csv"))

print("Weddings in our notebook (train):", len(train))
print("Weddings to predict (test):", len(test))
train.head()

Weddings in our notebook (train): 800
Weddings to predict (test): 400


,wedding_id,event,city,guests,aloo_count,mutton_kg,borhani_glasses,fairy_lights,drone_photographer,dhol_players,aunties_asking_when_marriage,went_back_for_seconds
0,W0001,Gaye Holud,Barishal,604,1327.0,138.1,529,478,no,6,8,0
1,W0002,Bou-bhat,Dhaka,166,104.0,35.0,172,445,yes,3,3,0
2,W0003,Biye,Mymensingh,67,59.0,16.7,72,116,yes,2,3,0
3,W0004,Biye,Dhaka,466,1206.0,122.3,668,1586,no,1,4,0
4,W0005,Biye,Khulna,287,209.0,81.9,278,646,yes,0,9,0


Each row is **one wedding**. The last column, `went_back_for_seconds`, is what we want to predict:
**1** = yes, guests went back for seconds and **0** = no.

The test file doesn't have this column. That's what *you* have to guess!

In [32]:
# How often do guests go back for seconds?
train["went_back_for_seconds"].value_counts()

went_back_for_seconds
0    464
1    336
Name: count, dtype: int64

In [ ]:
train.info()

##  Day 3: Tutul's mistakes

Okay, confession time. Tutul helped write the notebook and… there are problems.

1. Sometimes he wrote numbers in **Bangla digits** (like ৪৫০ instead of 450). Computers get confused by this!
2. At some weddings, the bride's uncle caught us and made us help serve food, so we **forgot to count the potatoes**. Those boxes are empty.
3. Tutul says he "maybe added an extra zero sometimes".

Real data scientists spend a LOT of time cleaning data. Let's fix the first one together.

In [33]:
# Fix 1: turn Bangla digits into English digits
numeric_cols=['guests','aloo_count','mutton_kg','borhani_glasses','fairy_lights','aunties_asking_when_marriage']

bangla_to_english = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")

def BN_to_EN(df, numeric_cols):
        for c in numeric_cols:
            df[c] = pd.to_numeric(
                df[c].astype(str).str.translate(bangla_to_english),errors="coerce",)
        return df
    
    
    #df["fairy_lights"] = df["fairy_lights"].astype(str).str.translate(bangla_to_english).astype(int)

train = BN_to_EN(train, numeric_cols)
test = BN_to_EN(test, numeric_cols)

print("Fixed!")

Fixed!


In [34]:
# Fix 2: how many empty boxes are there?
train.isna().sum()

wedding_id                       0
event                            0
city                             0
guests                           0
aloo_count                      24
mutton_kg                        0
borhani_glasses                  0
fairy_lights                     0
drone_photographer               0
dhol_players                     0
aunties_asking_when_marriage     0
went_back_for_seconds            0
dtype: int64

**Fix 3 is your mission.** Somewhere in `aloo_count` there are a few numbers that are 10 times too big.
Try `train.describe()` or sort the column with `train.sort_values("aloo_count")`. Can you spot Tutul's extra zeros?

In [8]:
train.describe()

,guests,aloo_count,mutton_kg,borhani_glasses,fairy_lights,dhol_players,aunties_asking_when_marriage,went_back_for_seconds
count,800.000000,776.000000,800.000000,800.000000,800.000000,800.000000,800.000000,800.000000
mean,371.623750,626.070876,94.350000,428.318750,970.915000,4.005000,5.996250,0.420000
std,184.408379,781.629417,49.023904,230.407713,600.886519,2.575513,2.491786,0.493867
min,51.000000,28.000000,9.300000,45.000000,81.000000,0.000000,1.000000,0.000000
25%,219.500000,262.750000,54.400000,241.000000,512.750000,2.000000,4.000000,0.000000
50%,367.000000,471.500000,92.900000,411.000000,821.500000,4.000000,6.000000,0.000000
75%,531.000000,850.000000,128.425000,597.000000,1305.500000,6.000000,8.000000,1.000000
max,700.000000,11520.000000,221.300000,1016.000000,2861.000000,8.000000,16.000000,1.000000


In [35]:
print(train["aloo_count"].sort_values(ascending=False).head(20))

501    11520.0
785    10270.0
671     8320.0
80      7150.0
447     6520.0
674     1831.0
556     1764.0
744     1739.0
677     1732.0
545     1712.0
196     1710.0
459     1684.0
13      1677.0
148     1665.0
678     1650.0
679     1602.0
641     1584.0
643     1553.0
289     1540.0
733     1537.0
Name: aloo_count, dtype: float64


In [36]:
print((test["aloo_count"] / test["guests"]).describe())
print((train["aloo_count"] / train["guests"]).describe())

count    400.000000
mean       1.529407
std        0.640855
min        0.402844
25%        0.953242
50%        1.535145
75%        2.069487
max        2.684466
dtype: float64
count    776.000000
mean       1.671515
std        1.764589
min        0.406015
25%        0.957247
50%        1.507989
75%        2.132390
max       26.752412
dtype: float64


In [37]:
train["ratio"] = train["aloo_count"] / train["guests"]
print(train["ratio"].sort_values(ascending=False).head(10))

671    26.752412
785    23.394077
80     22.698413
501    16.575540
447    16.464646
316     9.699248
115     2.700000
519     2.699758
28      2.698413
217     2.697168
Name: ratio, dtype: float64


In [38]:
# fix 10times bigger num

def alo_cleaning(df,max_ratio=3):
    df = df.copy()
    df["aloo_count"] = df["aloo_count"].where(df["ratio"] <= max_ratio, df["aloo_count"]/10)
    
    return df

train = alo_cleaning(train)

train = train.drop(columns="ratio", errors="ignore")

In [39]:
print((test["aloo_count"] / test["guests"]).describe())
print((train["aloo_count"] / train["guests"]).describe())

count    400.000000
mean       1.529407
std        0.640855
min        0.402844
25%        0.953242
50%        1.535145
75%        2.069487
max        2.684466
dtype: float64
count    776.000000
mean       1.537461
std        0.674965
min        0.406015
25%        0.957247
50%        1.506818
75%        2.127857
max        2.700000
dtype: float64


In [40]:
# Missing Values

median_ratio = (train["aloo_count"] / train["guests"]).median()

def Missing_values(df,median_ratio):
    miss = df["aloo_count"].isna()
    df.loc[miss, "aloo_count"] = (df.loc[miss, "guests"] * median_ratio).round()
    
    return df

train = Missing_values(train,median_ratio)
test = Missing_values(test,median_ratio)

In [46]:
train.head(5)

,wedding_id,event,city,guests,aloo_count,mutton_kg,borhani_glasses,fairy_lights,drone_photographer,dhol_players,aunties_asking_when_marriage,went_back_for_seconds
0,W0001,Gaye Holud,Barishal,604,1327.0,138.1,529,478,no,6,8,0
1,W0002,Bou-bhat,Dhaka,166,104.0,35.0,172,445,yes,3,3,0
2,W0003,Biye,Mymensingh,67,59.0,16.7,72,116,yes,2,3,0
3,W0004,Biye,Dhaka,466,1206.0,122.3,668,1586,no,1,4,0
4,W0005,Biye,Khulna,287,209.0,81.9,278,646,yes,0,9,0


## Day 4: My Algorithm Trading Cards

My ML teacher told me something that blew my mind: **most of these algorithms are older than my grandparents.** So I made trading cards. Collect them all!

| Card | Born | Inventor | Superpower | Weakness | Wedding use |
|---|---|---|---|---|---|
| **Linear Regression** | ~1805 | Legendre & Gauss | Draws the best straight line through data | Only thinks in straight lines | "Every guest drinks about 1 glass of borhani" |
| **Logistic Regression** | logistic curve 1838 | Verhulst (curve), later Berkson & Cox | Turns a straight line into a yes/no answer | Still one straight line inside | Yes/no: seconds or not? |
| **Decision Tree** | 1984 (CART) | Breiman, Friedman, Olshen & Stone | Plays 20 Questions with your data, and you can READ its rules | Gets confused if you let it grow too big | "Is it a big wedding? Is there enough aloo?" |
| **k-Nearest Neighbors** | 1951 | Fix & Hodges | "You are like your neighbors" | Slow when there's tons of data | Guests at the same table eat the same way |
| **Naive Bayes** | idea from 1763 | Thomas Bayes | Great at guessing with clues, powered early spam filters | Assumes every clue is independent | Bride's side or groom's side guest? |
| **Random Forest** | 2001 | Leo Breiman | Hundreds of trees vote together. Rafi's "MEGA AI"! | Can't imagine numbers bigger than it has seen | Rafi uses it for everything |

** Fun fact 1:** In 1801, the planet-hunter Carl Friedrich Gauss used the least-squares math behind linear regression to predict where the lost dwarf planet **Ceres** would show up in the sky, and astronomers found it there! If it can find a planet, it can find potatoes.

### Bangladesh Corner (special edition cards!)

** P. C. Mahalanobis**: one of the greatest statisticians ever. His family's roots go back to **Bikrampur**, in today's Munshiganj, Bangladesh. In 1936 he invented the **Mahalanobis distance**, a smart way of measuring how "far apart" things are, which ML people still use today.

** Satyendra Nath Bose**: while he was teaching at **Dhaka University**, he wrote a famous paper in 1924 and sent it to Albert Einstein. Today, a whole family of particles in the universe is called **bosons**, after him! 

## Day 5: Rafi's MEGA AI

Rafi said: *"Stop thinking. Just give ALL the columns to my MEGA AI. It's a Random Forest, which means it's HUNDREDS of trees. Hundreds! You can't beat hundreds."*

Fine, Rafi. Let's try it.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Turn "yes"/"no" into 1/0 so the computer understands
for df in (train, test):
    df["drone"] = (df["drone_photographer"] == "yes").astype(int)

rafi_columns = ["guests", "aloo_count", "mutton_kg", "borhani_glasses",
                "fairy_lights", "drone", "dhol_players", "aunties_asking_when_marriage"]

X = train[rafi_columns]
y = train["went_back_for_seconds"]

# Hide some weddings so we can test ourselves fairly
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.25, random_state=42)

mega_ai = RandomForestClassifier(n_estimators=300, random_state=42)
mega_ai.fit(X_tr, y_tr)

print("Rafi's MEGA AI accuracy:", round(accuracy_score(y_val, mega_ai.predict(X_val)) * 100, 1), "%")

In [ ]:
# Which columns does the MEGA AI think are important?
pd.Series(mega_ai.feature_importances_, index=rafi_columns).sort_values().plot.barh(title="What Rafi's MEGA AI cares about")
plt.show()

Rafi was SO proud. "See? It even uses the fairy lights!" 

Let's make a submission with it, so we have something on the leaderboard.

In [ ]:
mega_ai.fit(X, y)  # train on ALL our weddings this time
submission = pd.DataFrame({
    "wedding_id": test["wedding_id"],
    "went_back_for_seconds": mega_ai.predict(test[rafi_columns]),
})
submission.to_csv("submission.csv", index=False)
print("submission.csv is ready! Click 'Submit' on the competition page.")
submission.head()

## Day 6: Nodi asks "but WHY?"

Rafi's MEGA AI is okay. But it's not *great*. And Nodi won't stop asking questions:

> *"Rafi, if a wedding has 500 potatoes… is that a lot?"*

Rafi: "Yes! 500 is a lot!"

> *"What if there are 1,000 guests?"*

Rafi: "…oh."

Also, Babul Baburchi, the cook at the last wedding, told us something strange:
*"Too little aloo, people get sad. But too MUCH aloo? Then people think I saved money on the mutton!"*

Hmm. Hmm hmm hmm. 

**Your mission:** Make a NEW column that would make Nodi happy, then train a **tiny** decision tree (only 2 questions deep!) and see what happens.

<details>
<summary> Hint 1 (click me)</summary>
Big weddings have more of EVERYTHING: more aloo, more mutton, more borhani, more lights. Raw counts might be tricking us.
</details>

<details>
<summary> Hint 2 (only if you're really stuck!)</summary>
Imagine every guest gets their own plate. What matters is how much is on <b>each</b> plate. What maths operation gives you "per person"?
</details>

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

# YOUR TURN: make a new column. Replace the ____ with column names!
# train["my_new_column"] = train["____"] / train["____"]
# test["my_new_column"]  = test["____"]  / test["____"]

my_columns = ["guests"]  # change this to use your new column

tiny_tree = DecisionTreeClassifier(max_depth=2, random_state=42)
X_tr, X_val, y_tr, y_val = train_test_split(train[my_columns], y, test_size=0.25, random_state=42)
tiny_tree.fit(X_tr, y_tr)
print("Tiny tree accuracy:", round(accuracy_score(y_val, tiny_tree.predict(X_val)) * 100, 1), "%")

In [ ]:
# A decision tree is like a flowchart. You can READ what it learned!
plt.figure(figsize=(12, 6))
plot_tree(tiny_tree, feature_names=my_columns, class_names=["no seconds", "seconds!"], filled=True, rounded=True)
plt.show()

If you found the secret, your tiny tree should beat Rafi's hundreds of trees, and you can read the rule it found like a sentence.

Can you write the rule in your own words? Put it in your notebook! The best explanation wins the **Best Aloo Theory** prize.

In [ ]:
# When you're happy, make your own submission!
# tiny_tree.fit(train[my_columns], y)
# pd.DataFrame({
#     "wedding_id": test["wedding_id"],
#     "went_back_for_seconds": tiny_tree.predict(test[my_columns]),
# }).to_csv("submission.csv", index=False)

## Day 7: The finale

Here's the thing I didn't tell you: the test set includes the weddings from **the biggest wedding season ever**, including some HUGE weddings, bigger than anything in our notebook. Rafi's MEGA AI has never seen weddings that big… 

A model that understands the *real reason* guests go back for seconds doesn't care how big the wedding is.

**Things to try next:**
- Did you fix Tutul's extra zeros? Does it help?
- Try `LogisticRegression` or `KNeighborsClassifier` from the trading cards
- Does `mutton_kg` per guest matter too? Or is Rafi's fairy light theory right after all? 

Good luck, data scientist! Dhonnobad (thank you) for helping with my research. 

*- Mitu*